In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("netflix_titles.csv")

In [ ]:
display(df.head(10))
display(df.tail(5))

print("Shape:", df.shape)
print("Total cells:", df.size)

df.info(memory_usage="deep")

print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

print("Numerical columns:", df.select_dtypes(include="number").columns.tolist())
print("Categorical columns:", df.select_dtypes(include="object").columns.tolist())

for col in ["type", "rating", "release_year"]:
    print(f"Unique values in {col}:", df[col].nunique())

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

display(missing.sort_values("missing_percentage", ascending=False))

print("Duplicate rows:", df.duplicated().sum())

cleaned = df.drop_duplicates().copy()

cleaned["date_added"] = pd.to_datetime(
    cleaned["date_added"].str.strip(),
    errors="coerce"
)

cleaned["year_added"] = cleaned["date_added"].dt.year

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned.shape)

display(cleaned[["title", "director", "country", "date_added", "rating"]].head())


In [ ]:
movies_after_2015 = cleaned[
    (cleaned["type"] == "Movie") &
    (cleaned["release_year"] > 2015)
]

display(movies_after_2015[["title", "release_year", "rating"]].head(10))


cleaned["season_count"] = pd.to_numeric(
    cleaned["duration"].str.extract(r"(\d+)\s*Season")[0],
    errors="coerce"
)

long_tv_shows = cleaned[
    (cleaned["type"] == "TV Show") &
    (cleaned["season_count"] > 5)
]

display(long_tv_shows[["title", "duration", "season_count"]])


tv_ma = cleaned[cleaned["rating"] == "TV-MA"]

display(tv_ma[["title", "type", "rating"]].head(10))


india_content = cleaned[
    cleaned["country"].fillna("").str.contains("India", case=False) &
    (cleaned["release_year"] > 2018)
]

display(india_content[["title", "country", "release_year"]].head(10))


latest_titles = cleaned.sort_values(
    "date_added", ascending=False
)

display(latest_titles[["title", "type", "date_added"]].head(10))


love_titles = cleaned[
    cleaned["title"].str.contains("Love", case=False, na=False)
]

display(love_titles[["title", "type", "release_year"]].head(10))


filtered_titles = cleaned[
    (cleaned["type"] == "Movie") &
    (cleaned["release_year"] >= 2018) &
    (cleaned["rating"] == "TV-MA")
]

display(filtered_titles[["title", "release_year", "rating"]].head(10))


In [ ]:
cleaned["content_age"] = 2021 - cleaned["release_year"]


def classify_content(year):
    if pd.isna(year):
        return "Unknown"
    elif year >= 2018:
        return "Recent"
    elif year >= 2000:
        return "Modern"
    else:
        return "Classic"


cleaned["content_category"] = cleaned["release_year"].apply(
    classify_content
)


duration_parts = cleaned["duration"].str.extract(
    r"(?P<duration_value>\d+)\s*(?P<duration_unit>min|Season|Seasons)"
)

cleaned["duration_value"] = pd.to_numeric(
    duration_parts["duration_value"],
    errors="coerce"
)

cleaned["duration_unit"] = duration_parts["duration_unit"].replace({
    "min": "minutes",
    "Season": "seasons",
    "Seasons": "seasons"
})


movie_durations = cleaned[
    (cleaned["type"] == "Movie") &
    (cleaned["duration_unit"] == "minutes")
]

print("Average movie duration:",
      movie_durations["duration_value"].mean())


display(
    cleaned[cleaned["type"] == "TV Show"]
    .sort_values("season_count", ascending=False)
    [["title", "duration", "season_count"]]
    .head(10)
)


In [ ]:
display(cleaned["type"].value_counts())


titles_by_year = cleaned.groupby("year_added").size()

display(titles_by_year)



cleaned["primary_country"] = (
    cleaned["country"].str.split(", ").str[0]
)

top_countries = cleaned["primary_country"].value_counts().head(10)

display(top_countries)


country_type_counts = pd.crosstab(
    cleaned["primary_country"],
    cleaned["type"]
)

country_type_counts["Total"] = country_type_counts.sum(axis=1)

display(
    country_type_counts.sort_values("Total", ascending=False).head(10)
)


directors = (
    cleaned["director"]
    .dropna()
    .str.split(", ")
    .explode()
    .value_counts()
    .head(10)
)

display(directors)


average_release_year = cleaned.groupby("type")["release_year"].mean()

display(average_release_year)


common_ratings = cleaned.dropna(subset=["rating"]).groupby(
    "type"
)["rating"].agg(lambda x: x.mode().iloc[0])

display(common_ratings)


release_year_stats = cleaned.groupby("type")["release_year"].agg(
    ["mean", "median", "count"]
)

display(release_year_stats.round(2))


In [ ]:
pivot_year = pd.pivot_table(
    cleaned,
    index="year_added",
    columns="type",
    values="show_id",
    aggfunc="count",
    fill_value=0
)

display(pivot_year)


pivot_rating = pd.pivot_table(
    cleaned,
    index="rating",
    columns="type",
    values="show_id",
    aggfunc="count",
    fill_value=0
)

display(pivot_rating)


for content_type in cleaned["type"].dropna().unique():
    counts = cleaned.loc[
        cleaned["type"] == content_type, "rating"
    ].dropna().value_counts()

    print(content_type, ":", counts.index[0], "-", counts.iloc[0])


pivot_category = pd.pivot_table(
    cleaned,
    index="content_category",
    columns="type",
    values="show_id",
    aggfunc="count",
    fill_value=0
)

display(pivot_category)


In [ ]:
top_years = cleaned["year_added"].value_counts().head(5)

display(top_years)


selected_years = top_years.index

display(
    cleaned[cleaned["year_added"].isin(selected_years)]
    .pivot_table(
        index="year_added",
        columns="type",
        values="show_id",
        aggfunc="count",
        fill_value=0
    )
    .sort_index()
)


country_counts = cleaned["primary_country"].value_counts()

eligible_countries = country_counts[
    country_counts >= 100
].index

eligible_content = cleaned[
    cleaned["primary_country"].isin(eligible_countries)
]


country_mix = pd.crosstab(
    eligible_content["primary_country"],
    eligible_content["type"],
    normalize="index"
)

if "TV Show" in country_mix.columns:
    display(country_mix.sort_values("TV Show", ascending=False).head(10))


cleaned["release_period"] = np.select(
    [
        cleaned["release_year"] < 2000,
        cleaned["release_year"] >= 2000
    ],
    [
        "Before 2000",
        "2000 onward"
    ],
    default="Unknown"
)

display(pd.crosstab(cleaned["release_period"], cleaned["type"]))


latest_year = cleaned["year_added"].max()

print("Latest year:", latest_year)

display(
    cleaned.loc[
        cleaned["year_added"] == latest_year, "type"
    ].value_counts()
)


In [ ]:
sns.set_theme()


plt.figure(figsize=(7, 4))

sns.countplot(data=cleaned, x="type")

plt.title("Number of Movies and TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()


year_counts = (
    cleaned.dropna(subset=["year_added"])
    .groupby("year_added")
    .size()
    .sort_index()
)

plt.figure(figsize=(10, 4))

year_counts.plot(kind="line", marker="o")

plt.title("Netflix Titles Added by Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))

top_countries.sort_values().plot(kind="barh")

plt.title("Top 10 Countries by Number of Titles")
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


rating_data = cleaned.dropna(subset=["rating"])

plt.figure(figsize=(12, 5))

sns.countplot(
    data=rating_data,
    x="rating",
    hue="type",
    order=rating_data["rating"].value_counts().index
)

plt.title("Movies and TV Shows by Rating")
plt.xlabel("Rating")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



In [ ]:
genre_counts = (
    cleaned.dropna(subset=["listed_in"])
    .assign(
        genre=lambda x: x["listed_in"].str.split(", ")
    )
    .explode("genre")["genre"]
    .value_counts()
)

display(genre_counts.head(15))


plt.figure(figsize=(10, 5))

genre_counts.head(10).sort_values().plot(kind="barh")

plt.title("Top 10 Genres by Number of Titles")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()